In [1]:
!pip uninstall -y autogen pyautogen autogen-agentchat autogen-core autogen-ext flaml ray numpy
!pip install --no-cache-dir --force-reinstall numpy==1.26.4
!pip install --no-cache-dir pyautogen==0.2.25 flaml[automl] openai


Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 319.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 whi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 257.1/257.1 kB 323.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.6/223.6 MB 291.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 179.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 291.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.7/337.7 kB 336.2 MB/s eta 0:00:00
  Attempting uninstall: xgboost
    Found existing installation: xgboost 3.2.0
    Uninstalling xgboost-3.2.0:
      Successfully uninstalled xgboost-3.2.0


In [3]:
# ============================================================
# Scenario: Automated Business Report Assistant
# Problem Statement:
# Managers spend hours compiling business data from multiple sources.
# Reports need formatting, chart creation, and validation.
# This AutoGen-style workflow simulates:
# 1. Assistant Agent -> drafts report + Python code
# 2. UserProxy Agent -> executes code
# 3. Validation loop -> continues until final report is ready
# ============================================================

# -----------------------------
# Imports
# -----------------------------
import os
from google.colab import userdata

# NOTE:
# This code is written in classic AutoGen 0.2-style syntax.
# Use it only if your environment supports:
# pip install pyautogen==0.2.25

import autogen

# -----------------------------
# API Key Setup
# -----------------------------
groq_api_key = userdata.get("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = groq_api_key

# -----------------------------
# LLM Config
# -----------------------------
llm_config = {
    "config_list": [
        {
            "model": "llama-3.3-70b-versatile",
            "api_key": groq_api_key,
            "base_url": "https://api.groq.com/openai/v1",
            "api_type": "openai"
        }
    ],
    "temperature": 0,
    "timeout": 120
}
# -----------------------------
# Assistant Agent
# -----------------------------
assistant = autogen.AssistantAgent(
    name="Business_Report_Assistant",
    llm_config=llm_config,
    system_message="""
You are an enterprise business reporting assistant.

Your responsibilities:
1. Design a professional business report structure
2. Write Python code to:
   - simulate/fetch business data
   - clean and format it
   - generate summary statistics
   - create charts
   - export a polished report
3. Ensure the code is executable without errors
4. Add comments and maintain readability
5. If validation fails, fix and resend improved code
6. End the final successful response with DONE
"""
)

# -----------------------------
# User Proxy Agent
# -----------------------------
user_proxy = autogen.UserProxyAgent(
    name="Report_Executor",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=6,
    code_execution_config={
        "work_dir": "business_report_workspace",
        "use_docker": False
    },
    is_termination_msg=lambda msg: "DONE" in msg.get("content", ""),
    system_message="""
You are an automated execution and validation agent.

Your job:
1. Execute Python code generated by the assistant
2. Check whether:
   - data is processed correctly
   - charts are created
   - report file is saved
3. If there is any error, ask for correction
4. Approve only when the workflow is complete
"""
)

# -----------------------------
# Start Multi-Agent Workflow
# -----------------------------
user_proxy.initiate_chat(
    assistant,
    message="""
Build an automated business report assistant in Python.

Business Requirement:
Create a monthly sales performance report for a company using sample data.

Expected Workflow:
1. Simulate business data from multiple sources such as CRM, ERP, and spreadsheets
2. Merge and clean the data
3. Calculate:
   - total sales
   - average sales
   - highest sales region
   - lowest sales region
   - monthly growth trend
4. Generate at least 2 charts:
   - bar chart for region-wise sales
   - line chart for monthly sales trend
5. Create a polished text report and save it as business_report.txt
6. Ensure the output is professional and readable
7. Include validation-friendly print statements

Use Python only.
Make the code complete and executable.
End only when the task is fully complete with DONE.
"""
)

Report_Executor (to Business_Report_Assistant):


Build an automated business report assistant in Python.

Business Requirement:
Create a monthly sales performance report for a company using sample data.

Expected Workflow:
1. Simulate business data from multiple sources such as CRM, ERP, and spreadsheets
2. Merge and clean the data
3. Calculate:
   - total sales
   - average sales
   - highest sales region
   - lowest sales region
   - monthly growth trend
4. Generate at least 2 charts:
   - bar chart for region-wise sales
   - line chart for monthly sales trend
5. Create a polished text report and save it as business_report.txt
6. Ensure the output is professional and readable
7. Include validation-friendly print statements

Use Python only.
Make the code complete and executable.
End only when the task is fully complete with DONE.


--------------------------------------------------------------------------------
Business_Report_Assistant (to Report_Executor):

```python
# Import nece

ChatResult(chat_id=None, chat_history=[{'content': '\nBuild an automated business report assistant in Python.\n\nBusiness Requirement:\nCreate a monthly sales performance report for a company using sample data.\n\nExpected Workflow:\n1. Simulate business data from multiple sources such as CRM, ERP, and spreadsheets\n2. Merge and clean the data\n3. Calculate:\n   - total sales\n   - average sales\n   - highest sales region\n   - lowest sales region\n   - monthly growth trend\n4. Generate at least 2 charts:\n   - bar chart for region-wise sales\n   - line chart for monthly sales trend\n5. Create a polished text report and save it as business_report.txt\n6. Ensure the output is professional and readable\n7. Include validation-friendly print statements\n\nUse Python only.\nMake the code complete and executable.\nEnd only when the task is fully complete with DONE.\n', 'role': 'assistant'}, {'content': '```python\n# Import necessary libraries\nimport pandas as pd\nimport numpy as np\nimport 